In [1]:
import numpy as np
import random

In [2]:
def reorder_adjacency_matrix(adjacency_matrix,diag_cut=True):
    original = adjacency_matrix
    if diag_cut:
        for i in range(adjacency_matrix.shape[0]):
            original[i][i]=0#edges from i to i don't contribute to the solution, so we will ignore them
    #we want to sort the matrix by the amount of connections the rows have. ofc this has to happen
    #in the same way with the columns.
    original_order = np.arange(adjacency_matrix.shape[0],dtype=np.int32)#this is actually redundant because we have the index.
    post_order = np.argsort(adjacency_matrix.sum(axis=1,dtype=np.int64))[::-1]#this is the index-set we will use
    amount_post_order = np.sort(adjacency_matrix.sum(axis=1,dtype=np.int32))[::-1]#these are the numbers we will use to replace the 1's

    
    sorted_matrix = original[post_order,:][:,post_order].copy()#now the row with the most connections is at the top
    #while rows with less connections follow in descending order
    #the cols are sorted too, so we don't have to worry about translation errors.
    
    return sorted_matrix, original_order, post_order, amount_post_order# we'll need all of them

In [423]:
def select_and_reassign_significant_part(sorted_adjacency_matrix,amount_post_order):
    sorted_adjacency_matrix = sorted_adjacency_matrix.copy()
    amount_post_order = amount_post_order.copy()
    for i, row in enumerate(sorted_adjacency_matrix):
        sorted_adjacency_matrix[i]=row*amount_post_order
    last_non_trivial_row = (amount_post_order>=2).sum()
    idx = np.arange(last_non_trivial_row,dtype=np.int32)
    sorted_adjacency_matrix = sorted_adjacency_matrix[idx,:][:,idx]
    return np.array(sorted_adjacency_matrix,dtype=np.int32), amount_post_order[:last_non_trivial_row]

In [424]:
def compute_incoming_connections(adjacency_matrix):
    incoming_connections = (adjacency_matrix!=0).sum(axis=0)
    sorted_numerated_in_matrix = (adjacency_matrix!=0)*incoming_connections
    return sorted_numerated_in_matrix,incoming_connections

In [425]:
class CliqueGenerator():
    """
    Create an adjacency matrix with cliques in it to test our algorithm. It will have 'num_cliques' 
    cliques with random size between 3 and max_clique_size.
    This function returns the adjacency matrix, the solutions and the sizes of the cliques.
    """
    def __init__(self, 
                 noise_level, 
                 adjacency_matrix_size, 
                 num_cliques,
                 max_clique_size, 
                 info=False,
                diag_cut=False):
        
        self.noise_level = noise_level
        self.adjacency_matrix_size = adjacency_matrix_size
        self.num_cliques = num_cliques
        self.info = info
        self.max_clique_size = max_clique_size
        
    def create(self,num_cliques=None,diag_cut=True):
        if num_cliques is None:
            num_cliques = self.num_cliques
        assert num_cliques <= self.adjacency_matrix_size
        placeholder_array = np.zeros((self.adjacency_matrix_size,self.adjacency_matrix_size))
        clique_sizes = []
        for i in range(num_cliques):
            clique_sizes.append(int(np.random.randint(3,self.max_clique_size)))
        
        candidates = []
        i = 0
        while len(candidates)<num_cliques:
            candidate = []
            while len(candidate) < clique_sizes[i]:
                tmp = int(np.random.randint(0,self.adjacency_matrix_size))
                if tmp not in candidate:
                    candidate.append(tmp)
                candidate.sort()
            if candidate not in candidates:
                candidates.append(candidate)
                i+=1
        
        
        noise = int((1-self.noise_level)*100)
        noise_matrix = np.random.randint(0,100,size=(self.adjacency_matrix_size,self.adjacency_matrix_size))>noise
        difference = noise_matrix-placeholder_array*noise_matrix
        placeholder_array += difference
        for candidate in candidates:
            for i in candidate:
                for j in candidate:
                    if j!=i:
                        placeholder_array[i][j]=1
        if self.info:
            print("[",len(clique_sizes),"]","Cliques will be created in the following sizes: ",*clique_sizes,"\n\n","-"*30)
            print("The solutions will be: ", candidates,"\n\n","-"*30,"\n")  
            print("Now we have a pure adjacency matrix without any noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)


        
        
        if self.info:
            print("\n\n","-"*30,"\nNow we have a pure adjacency matrix with noise:","\n\n","-"*30,"\n",placeholder_array,"\n\n","-"*30)
        if diag_cut:
            for i in range(placeholder_array.shape[0]):
                placeholder_array[i][i]=0#edges from i to i don't contribute to the solution, so we will ignore them
        
        return placeholder_array, candidates, clique_sizes

# Bis hierher alles sauber.

In [446]:
class Solver():
    def __init__(self,
                 noise_level = 0.05, 
                 adjacency_matrix_size = 20, 
                 num_cliques = 5, 
                 max_clique_size =5,
                 adjacency_matrix=None,
                 info=False,
                 diag_cut=True,
                print_solutions=True):
        
        self.diag_cut = diag_cut
        if adjacency_matrix is None:
            self.gen = CliqueGenerator(noise_level = noise_level, 
                                   adjacency_matrix_size = adjacency_matrix_size, 
                                   num_cliques = num_cliques, 
                                   max_clique_size = max_clique_size,
                                   info=info)
        
        
            self.adjacency_matrix, self.solutions, self.clique_sizes = self.gen.create()
            if print_solutions:
                print("Cliques at: ",self.solutions)
        else:
            self.adjacency_matrix = adjacency_matrix
        
        #from here on, only adjacency matrix is used. the code above is just to give you an example
        
        self.sorted_adjacency_matrix, self.original_order, self.post_order, self.amount_post_order = reorder_adjacency_matrix(self.adjacency_matrix,diag_cut=diag_cut)
        self.current_order = self.post_order
        self.sorted_numerated_out_matrix, self.outgoing_connections = select_and_reassign_significant_part(self.sorted_adjacency_matrix,self.amount_post_order)
        self.sorted_numerated_in_matrix, self.incoming_connections = compute_incoming_connections(self.sorted_numerated_out_matrix)
        self.valid_index = np.arange(self.sorted_numerated_out_matrix.shape[0])
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        self.current_order = self.current_order[:len(self.outgoing_connections)]
        self.current_order = self.current_order[self.nodes_of_interest]
        self.sorted_numerated_out_matrix=self.sorted_numerated_out_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.sorted_numerated_in_matrix=self.sorted_numerated_in_matrix[self.nodes_of_interest,:][:,self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.exclusion_prep = self.post_order[:len(self.outgoing_connections)]
        self.not_included = self.exclusion_prep[~self.nodes_of_interest]
        self.exclusion_prep = self.exclusion_prep[self.nodes_of_interest]
        self.outgoing_connections = self.outgoing_connections[self.nodes_of_interest]
        self.incoming_connections = self.incoming_connections[self.nodes_of_interest]
        self.nodes_of_interest = self.nodes_of_interest[self.nodes_of_interest]
        self.valid_index=self.valid_index[self.nodes_of_interest]
        self.helper_adjacency_matrix = self.create_helper()
        self.cliques = []
        
        self.added_for_removal = []
        
        self.update_nodes_of_interest()
        
    def info(self):
        print("self.sorted_numerated_out_matrix",self.sorted_numerated_out_matrix.shape)
        print("self.sorted_numerated_in_matrix",self.sorted_numerated_in_matrix.shape)
        print("self.valid_index",self.valid_index.shape)
        print("self.exclusion_prep",self.exclusion_prep.shape)
        print("self.outgoing_connections",self.outgoing_connections.shape)
        print("self.incoming_connections",self.incoming_connections.shape)
        print("self.nodes_of_interest",self.nodes_of_interest.shape)
        print("self.valid_index",self.valid_index.shape)
        print("self.current_order",self.current_order.shape)
        print("self.post_order: ",self.post_order)
        print("self.current_order: ",self.current_order)
        print("self.outgoing_connections: ",self.outgoing_connections)
        print("self.incoming_connections: ",self.incoming_connections)
        print("-"*60,"\nself.sorted_numerated_out_matrix\n")
        self.out_matrix()
        print("-"*60,"\nself.sorted_numerated_in_matrix\n")
        self.in_matrix()
        print("-"*60)
        print("Exclusion Prep: ",self.exclusion_prep,"Not included:", self.not_included)
        
    def out_matrix(self):
        print(self.sorted_numerated_out_matrix,"\n",self.sorted_numerated_out_matrix.shape,"\n")
    def in_matrix(self):
        print(self.sorted_numerated_in_matrix,"\n",self.sorted_numerated_in_matrix.shape,"\n")

        
    # Bis hierher ok.
    def correct_out(self,col):
        mask = self.sorted_numerated_out_matrix[:,col]!=0
        self.sorted_numerated_out_matrix[:,col]-=mask
        self.outgoing_connections[col]-=1
        
    def correct_in(self,col):
        mask = self.sorted_numerated_in_matrix[:,col]!=0
        self.sorted_numerated_in_matrix[:,col]-=mask
        self.incoming_connections[col]-=1
    
    def reduce_single_in(self,col,row):
        self.sorted_numerated_in_matrix[col,row]-=1
    
    def reduce_single_out(self,col,row):
        self.sorted_numerated_out_matrix[col,row]-=1
        
    def remove_idx(self,idx):
        assert self.sorted_numerated_out_matrix.shape[0]>0
        idx_len = self.sorted_numerated_out_matrix.shape[0]
        valid_idx = np.ones(idx_len,dtype=np.int32)
        valid_idx[idx]=0
        valid_idx = np.bool(valid_idx)
        for i,b in enumerate(self.sorted_numerated_out_matrix[idx]!=0):
            if b:
                self.correct_in(i)
        self.sorted_numerated_out_matrix = self.sorted_numerated_out_matrix[valid_idx,:][:,valid_idx]
        self.sorted_numerated_in_matrix = self.sorted_numerated_in_matrix[valid_idx,:][:,valid_idx]
        
                
        self.not_included.append(list(self.exclusion_prep[~valid_idx]))
        self.exclusion_prep = self.exclusion_prep[valid_idx]
        self.outgoing_connections = self.outgoing_connections[valid_idx]
        self.incoming_connections = self.incoming_connections[valid_idx]
        self.current_order = self.current_order[valid_idx]
        self.nodes_of_interest = self.nodes_of_interest[valid_idx]
    
    def remove_job(self):#do this ONLY if everything else is done and you are tired of your collected data
        for i in self.added_for_removal:
            self.remove_idx(i)
        self.added_for_removal = []
    
    def update_nodes_of_interest(self,delete=True):# same here, ackthuallhy it became the only and safe instance where you can do it
        self.nodes_of_interest = (self.outgoing_connections>1)&(self.incoming_connections>1)
        mask = ~self.nodes_of_interest
        for i,b in enumerate(mask):
            if b:
                if i not in self.added_for_removal:
                    self.added_for_removal.append(i)
        if delete:
            self.remove_job()
            
    def final_check_if_clique(self,real_indeces):#
        check = np.ones(len(real_indeces),dtype=np.int32)
        for count,i in enumerate(real_indeces):
            for j in real_indeces:
                if i!=j:
                    check[count] = check[count]==self.adjacency_matrix[i][j]
        return check.sum()==len(real_indeces)
    
    def set_inf_clique(self,real_indices):#
        for i in real_indices:
            for j in real_indices:
                if i!=j:
                    self.adjacency_matrix[i][j]=np.inf
    
    def translator(self, candidates):#
        rgw = []
        for i in candidates:
            rgw.append(self.current_order[i].item())
        return np.array(rgw,dtype=np.int32)
    
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)            
    
    def create_helper(self):
        self.helper_adjacency_matrix = self.sorted_numerated_in_matrix != 0
        
    # Bis hierher absolut alles okay.    


    
    def check_idx_part1(self,idx):#
        assert idx<len(self.outgoing_connections)# just to make sure we are in the right intervall
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
            
        
        lst = self.helper_adjacency_matrix[idx]
        
        sum_out = lst.sum()
        #print("sum_out before: ",sum_out)
        if sum_out <2:
            if idx not in self.added_for_removal:
                self.added_for_removal.append(idx)
            return idx, 0, [], {}
        check = 0           #here : counter for suitable candidates
        indices = []        #for the outer matrix
        
        for i,b in enumerate(lst):
            if b:
                if self.helper_adjacency_matrix[i][idx]:
                    indices.append(i)
                    check+=1
                else:
                    #self.sorted_numerated_out_matrix[i][idx]-=1 # legacy?
                    sum_out-=1
        tmp_bool_dict={}#to avoid new calculations
        if check<2:#this means it can never be a part of a clique, to reduce compute delete it.
            if idx not in self.added_for_removal:
                self.added_for_removal.append(idx)
            return idx, 0, [], {}

        for i in indices:
            tmp_bool_dict[i]=self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[i]
        return idx, check, indices, tmp_bool_dict
                    
                    
    def check_idx_part2(self,idx,check,indices,tmp_bool_dict):#
        check=check
        candidates = []
        for index in reversed(indices):
            tmp_bool_sum = tmp_bool_dict[index].sum()
            if tmp_bool_sum>0:
                tmp_bool_vec2 = tmp_bool_dict[index]
                tmp_bool_vec2[idx]=1
                tmp_bool_vec2[index]=1
                count = len(self.helper_adjacency_matrix[0])-1
            else:
                continue
                
            while tmp_bool_sum and count>=0:
                if tmp_bool_dict[index][count]==1 and count!=index and count!=idx:
                    tmp_bool_sum-=1
                    if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:#this may look like we iterate over the whole matrix, but we only take the ones
                        if count not in candidates and count!=idx and count!=index:
                            candidates.append(count)
                        else:
                            self.correct_in(idx)
                if index not in candidates:
                    candidates.append(index)
                count-=1
            check-=1
            if check<2 and len(candidates)<1:#clique begins with 3
                if idx not in self.added_for_removal:
                    self.added_for_removal.append(idx)
                return
        return candidates
    
    def check_idx_part3(self,idx,candidates):#
        candidates=candidates
        if idx not in candidates:
            candidates.append(idx)
        checkbox = np.zeros((len(candidates),len(candidates)))
        for i in range(len(candidates)-1):
            for j in range(len(candidates)-1):
                checkbox[i][j]=self.helper_adjacency_matrix[candidates[i]][candidates[j]]!=0
        checkbox[len(candidates)-1][len(candidates)-1]=1
        checkbox = np.array(checkbox,dtype=np.int32)
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        return checkbox, real_candidates
    
    
    
    
    # addieren der box + multiplizieren mit idx GEILER CHECK
    # helper matrix, nicht adjacency matrix
    # info aujs out und in reinholen welche maximale clique überhaupt möglich sit bevor man überhaupt testet,
    #dafür ganze zeile holen von idx, reicht ja. dann min zwischen in und out
    #beginne die summation mit dem höchsten aber erst dann, wenn die anzahl gleich der höhe ist, nicht davor
    def check_idx_part4(self):
        pass
    
    
    def check_how_many_cliques_possible(self,candidates,real_indeces,idx):
        print("candidates:",candidates)
        print("real_indeces:",real_indeces)
        print("idx:",idx)
        print("self.out_matrix[idx]: ",self.sorted_numerated_out_matrix[idx])
        print("self.in_matrix[idx]: ",self.sorted_numerated_in_matrix[idx])
        print("self.correct_idx:",self.current_order)
        #########################################################################
        
        pass
    


    
    def check_idx(self,idx):
        print("idx:",idx)
        idx, check, indices, tmp_bool_dict = self.check_idx_part1(idx)
        candidates = self.check_idx_part2(idx, check, indices, tmp_bool_dict)
        checkbox, real_candidates = self.check_idx_part3(idx,candidates)
        
        candidates2=candidates
        if idx not in candidates2:
            candidates2.append(idx)
        real_indeces = self.translator(candidates)
        
        
        
        
        self.check_how_many_cliques_possible(candidates,real_indeces,idx)

        
        """
        print("checkbox: ",checkbox)
        print("checkbox.sum(axis=1): ",checkbox.sum(axis=1))
        print("candidates: ",candidates)
        print("self.current_order[candidates]: ",self.current_order[candidates])
        print("self.adjacency_matrix[self.current_order[candidates]]: ",self.adjacency_matrix[self.current_order[candidates]])
        print("self.added_for_removal: ",self.added_for_removal)
        print("#################################")
        """
        
        #self.remove_job()
        #self.update_nodes_of_interest()

In [447]:
# ne dreierclique addiert exakt 2 auf term von cliquenmitglied. eine viererclique addiert exakt 3. also immer k-1
# vielleicht lässt sich das irgendwie verwenden.

In [448]:
S = Solver()
#self.current_order ist the magic list


Cliques at:  [[4, 10, 11, 13], [4, 14, 18], [0, 3, 9], [4, 5, 7, 14], [3, 6, 7, 8]]


In [449]:
S.check_idx(5)

idx: 5
candidates: [7, 6, 0, 5]
real_indeces: [11 10  4 13]
idx: 5
self.out_matrix[idx]:  [8 0 0 0 0 0 4 4 3 0 0 0 0]
self.in_matrix[idx]:  [7 0 0 0 0 0 3 4 5 0 0 0 0]
self.correct_idx: [ 4  7  3 14  8 13 10 11  5  0  6 18  9]


In [452]:
S.final_check_if_clique(np.array([4, 10, 3, 13],dtype=np.int32))

np.False_

In [437]:
S.info()

self.sorted_numerated_out_matrix (12, 12)
self.sorted_numerated_in_matrix (12, 12)
self.valid_index (12,)
self.exclusion_prep (12,)
self.outgoing_connections (12,)
self.incoming_connections (12,)
self.nodes_of_interest (12,)
self.valid_index (12,)
self.current_order (12,)
self.post_order:  [17  0  9  7 10 16 19 18  5  1  8 12  6  3 11 15 14 13  4  2]
self.current_order:  [17  0  9  7 10 16 19 18  5  1  8 12]
self.outgoing_connections:  [6 6 5 5 5 3 3 3 3 3 3 2]
self.incoming_connections:  [5 5 6 6 4 1 2 3 4 2 2 1]
------------------------------------------------------------ 
self.sorted_numerated_out_matrix

[[0 6 5 5 0 3 0 0 0 0 0 2]
 [6 0 5 5 0 0 0 3 3 0 0 0]
 [6 6 0 5 5 0 3 0 0 0 0 0]
 [6 6 5 0 0 0 0 3 3 0 0 0]
 [0 0 5 5 0 0 3 0 0 3 3 0]
 [6 0 0 0 0 0 0 0 0 0 0 2]
 [0 0 5 0 5 0 0 0 0 0 0 0]
 [0 6 0 5 0 0 0 0 3 0 0 0]
 [0 6 0 5 0 0 0 3 0 0 0 0]
 [0 0 5 0 5 0 0 0 0 0 3 0]
 [0 0 0 0 5 0 0 0 3 3 0 0]
 [6 0 0 0 0 3 0 0 0 0 0 0]] 
 (12, 12) 

----------------------------------------------

In [219]:
A = {}

In [220]:
A[1]=[1,2,3]

In [222]:
A[1][2]

3

In [ ]:

        else:
            for index in reversed(indices): # we begin with the row that has the least connections, which is  the rightmost row
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                if tmp_bool_sum>1:
                    tmp_bool_dict[index]=tmp_bool_vec#safe it for later
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1#since self connections were crossed out
                    tmp_bool_vec2[index]=1
                    
                    negative_count = len(self.helper_adjacency_matrix[0])-1#we start from right again
                    
                    while tmp_bool_sum>0 and negative_count>=0:
                        
                    

In [ ]:
    def check_idx(self,idx):
        assert idx<len(self.outgoing_connections)
        print("1) idx",idx)
        print("#"*40)
        #to avoid repetetive row computations
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
        print("2) helper_adjacency_matrix\n",self.helper_adjacency_matrix,"\n len(self.outgoing_connections): ",len(self.outgoing_connections))
    
        print("#"*40)
        lst = self.helper_adjacency_matrix[idx]#get the specific row we want to work on
        print("lst: self.helper_adjacency_matrix[idx]",lst)
        sum_out = lst.sum()#amount of outgoing connections
        print("sum_out: lst.sum(): ",sum_out)
        check = 0#here : counter for suitable candidates
        indices = []#for the outer matrix
        candidates = []#if a node is suitable, it will appear here
        #print(lst)
        print("-"*40)

        print("check = 0, indices = [], candidates = []\n\nbegin for i,b in enumerate(lst)")
        for i,b in enumerate(lst):
            if b:
                if sum_out<2:
                    print("if b: if sum_out < 2: break")
                    break
                if self.helper_adjacency_matrix[i][idx]:
                    print("ansonsten : wenn self.sorted_numerated_out_matrix[i][idx] ungleich 0: dann ##",self.sorted_numerated_out_matrix[i][idx])
                    indices.append(i)
                    check+=1
                    print("indices append, check+=1")
                else:
                    sum_out-=1
                    print("else sum_out-=1")
        if check<2:
            print("Wenn Check<2 dann self.remove_idx(idx), sollte hier das ende sein.")
            self.remove_idx(idx)#das richtig
            return
        else:
            print("%"*40)

            print("Check offensichtlich größer als 2")
            print("begin der reversed(indices) schleife. für index in ",reversed(indices))
            for index in reversed(indices):#begin with the smallest. the indices with the most connections are to the right
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                print("tmp_bool_vec ",tmp_bool_vec)
                print("tmp_bool_sum ",tmp_bool_sum)
                if tmp_bool_sum>1:
                    print("$"*40)
                    print("Da tmp_bool_sum > 1: ")
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1
                    tmp_bool_vec2[index]=1
                    print(f"tmp_bool_vec2 wurde befüllt an {idx} und {index}: ",tmp_bool_vec2)
                    count=len(self.helper_adjacency_matrix[0])-1
                    print("count=len(self.helper_adjacency_matrix[0])-1",count)
                    print("."*40)
                    while tmp_bool_sum and count>=0:
                        print("while tmp_bool_sum and count>=0:")
                        if tmp_bool_vec[count]==1 and count!=index and count!=idx:
                            tmp_bool_sum-=1
                            print("if tmp_bool_vec[count]==1 and count!=index and count!=idx: tmp_bool_sum-=1: ",tmp_bool_sum)
                            if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:
                                print("if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:")
                                if count not in candidates and count!=idx and count!=index:
                                    print("if count not in candidates and count!=idx and count!=index: candidates.append(count)")
                                    candidates.append(count)
                                    if index not in candidates:
                                        candidates.append(index)#does basically the same, just marking it
                                else:
                                    print("ELSE: self.correct_in(idx)")
                                    self.correct_in(idx)
                        
                        count-=1
                        print("unterstes ende while: count", count)
                        print("."*40)
                    check-=1
                    print("+"*40)

                    if check<2 and len(candidates)<1:#clique begins with 3
                        self.remove_idx(idx)
                        print("Wenn check<2 und len(candidates)<1, dann remove_idx(idx) - normal, hier gibts nichts zu sehen")
                        return
                    print("+"*40)

                #print(candidates)
                #print("idx: ",idx)
                #print("index",index)
                for c in candidates:
                    if c in self.not_included:
                        candidates.remove(c)
        print("?"*40)
        
        candidates.sort()# because again, low indeces mean less connections, this is where we wlll start to eliminate
        print("Wir sind außerhalb der while. sortieren jetzt candidates um von rechts nach links zu arbeiten ",candidates,"\n\n")
        if idx not in candidates:
            candidates.append(idx)#we want it to be the most right one
        print("Wir appenden idx an candidates: ",candidates)
        checkbox = np.zeros((len(candidates),len(candidates)))
        
        for i in range(len(candidates)):
            for j in range(len(candidates)):
                print("HIIIIIER:")
                print("self.sorted_numerated_out_matrix: ",self.sorted_numerated_out_matrix.shape)
                print("self.current_order: ",len(self.current_order))
                print("candidates: ",len(candidates))
                print("")
                checkbox[i][j]=self.helper_adjacency_matrix[self.current_order[candidates[i]]][self.current_order[candidates[j]]]!=0
                
            
        checkbox[len(candidates)-1][len(candidates)-1]=1
        checkbox = np.array(checkbox,dtype=np.int32)
    
        print("jetzt haben wir ne Checkbox mit den richtigen EInträgen: \n",checkbox)
        
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        print("\n Und wir haben real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32): ",real_candidates)
        not_suitable_candidates = []
        #for c in candidates[real_candidates]:
            #not_suitable_candidates.append(c)
        #candidates = candidates[real_candidates]
        #print(real_candidates)

        print("?"*40)
        print("&"*40)

        length = len(real_candidates)
        print("Die Länge davon: length = ",length)
        for i,candidate in enumerate(reversed(candidates)):
            c=length-i-1
            if real_candidates[c]:
                
                print(checkbox[:,c])
                if c == length-1:
                    continue
                print(len(checkbox[:,c]),checkbox[:,c])
                print(len(checkbox[:,length-1]),checkbox[:,length-1])
                check = checkbox[:,c]&checkbox[:,length-1]
                check_sum = (check).sum()
                print(check, check_sum)
                if check_sum < 1:
                    print("idx: ",candidates[i])
                    
        print("&"*40)
        print("Insgesamt wäre das mapping: ")
        for i in candidates:
            print(i," : ",self.outgoing_connections[i])

        #self.remove_idx(idx)
        self.update_nodes_of_interest()#at the end     
        
        
        
        
        
        
        
        
        
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)

In [ ]:
    def check_idx(self,idx):
        assert idx<len(self.outgoing_connections)
        print("1) idx",idx)
        print("#"*40)
        #to avoid repetetive row computations
        if self.helper_adjacency_matrix is None:
            self.create_helper()
        if len(self.outgoing_connections)!=len(self.helper_adjacency_matrix[0]):
            self.create_helper()
        print("2) helper_adjacency_matrix\n",self.helper_adjacency_matrix,"\n len(self.outgoing_connections): ",len(self.outgoing_connections))
    
        print("#"*40)
        lst = self.helper_adjacency_matrix[idx]#get the specific row we want to work on
        print("lst: self.helper_adjacency_matrix[idx]",lst)
        sum_out = lst.sum()#amount of outgoing connections
        print("sum_out: lst.sum(): ",sum_out)
        check = 0#here : counter for suitable candidates
        indices = []#for the outer matrix
        candidates = []#if a node is suitable, it will appear here
        #print(lst)
        print("-"*40)

        print("check = 0, indices = [], candidates = []\n\nbegin for i,b in enumerate(lst)")
        for i,b in enumerate(lst):
            if b:
                if sum_out<2:
                    print("if b: if sum_out < 2: break")
                    break
                if self.helper_adjacency_matrix[i][idx]:
                    print("ansonsten : wenn self.sorted_numerated_out_matrix[i][idx] ungleich 0: dann ##",self.sorted_numerated_out_matrix[i][idx])
                    indices.append(i)
                    check+=1
                    print("indices append, check+=1")
                else:
                    sum_out-=1
                    print("else sum_out-=1")
        if check<2:
            print("Wenn Check<2 dann self.remove_idx(idx), sollte hier das ende sein.")
            self.remove_idx(idx)#das richtig
            return
        else:
            print("%"*40)

            print("Check offensichtlich größer als 2")
            print("begin der reversed(indices) schleife. für index in ",reversed(indices))
            for index in reversed(indices):#begin with the smallest. the indices with the most connections are to the right
                tmp_bool_vec = self.helper_adjacency_matrix[idx]&self.helper_adjacency_matrix[index]
                tmp_bool_sum = tmp_bool_vec.sum()
                print("tmp_bool_vec ",tmp_bool_vec)
                print("tmp_bool_sum ",tmp_bool_sum)
                if tmp_bool_sum>1:
                    print("$"*40)
                    print("Da tmp_bool_sum > 1: ")
                    tmp_bool_vec2 = tmp_bool_vec
                    tmp_bool_vec2[idx]=1
                    tmp_bool_vec2[index]=1
                    print(f"tmp_bool_vec2 wurde befüllt an {idx} und {index}: ",tmp_bool_vec2)
                    count=len(self.helper_adjacency_matrix[0])-1
                    print("count=len(self.helper_adjacency_matrix[0])-1",count)
                    print("."*40)
                    while tmp_bool_sum and count>=0:
                        print("while tmp_bool_sum and count>=0:")
                        if tmp_bool_vec[count]==1 and count!=index and count!=idx:
                            tmp_bool_sum-=1
                            print("if tmp_bool_vec[count]==1 and count!=index and count!=idx: tmp_bool_sum-=1: ",tmp_bool_sum)
                            if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:
                                print("if (self.helper_adjacency_matrix[count]&tmp_bool_vec2).sum()>=2:")
                                if count not in candidates and count!=idx and count!=index:
                                    print("if count not in candidates and count!=idx and count!=index: candidates.append(count)")
                                    candidates.append(count)
                                    if index not in candidates:
                                        candidates.append(index)#does basically the same, just marking it
                                else:
                                    print("ELSE: self.correct_in(idx)")
                                    self.correct_in(idx)
                        
                        count-=1
                        print("unterstes ende while: count", count)
                        print("."*40)
                    check-=1
                    print("+"*40)

                    if check<2 and len(candidates)<1:#clique begins with 3
                        self.remove_idx(idx)
                        print("Wenn check<2 und len(candidates)<1, dann remove_idx(idx) - normal, hier gibts nichts zu sehen")
                        return
                    print("+"*40)

                #print(candidates)
                #print("idx: ",idx)
                #print("index",index)
                for c in candidates:
                    if c in self.not_included:
                        candidates.remove(c)
        print("?"*40)
        
        candidates.sort()# because again, low indeces mean less connections, this is where we wlll start to eliminate
        print("Wir sind außerhalb der while. sortieren jetzt candidates um von rechts nach links zu arbeiten ",candidates,"\n\n")
        if idx not in candidates:
            candidates.append(idx)#we want it to be the most right one
        print("Wir appenden idx an candidates: ",candidates)
        checkbox = np.zeros((len(candidates),len(candidates)))
        
        for i in range(len(candidates)):
            for j in range(len(candidates)):
                print("HIIIIIER:")
                print("self.sorted_numerated_out_matrix: ",self.sorted_numerated_out_matrix.shape)
                print("self.current_order: ",len(self.current_order))
                print("candidates: ",len(candidates))
                print("")
                checkbox[i][j]=self.helper_adjacency_matrix[self.current_order[candidates[i]]][self.current_order[candidates[j]]]!=0
                
            
        checkbox[len(candidates)-1][len(candidates)-1]=1
        checkbox = np.array(checkbox,dtype=np.int32)
    
        print("jetzt haben wir ne Checkbox mit den richtigen EInträgen: \n",checkbox)
        
        real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32)
        print("\n Und wir haben real_candidates = np.array(checkbox.sum(axis=0)>1,dtype=np.int32): ",real_candidates)
        not_suitable_candidates = []
        #for c in candidates[real_candidates]:
            #not_suitable_candidates.append(c)
        #candidates = candidates[real_candidates]
        #print(real_candidates)

        print("?"*40)
        print("&"*40)

        length = len(real_candidates)
        print("Die Länge davon: length = ",length)
        for i,candidate in enumerate(reversed(candidates)):
            c=length-i-1
            if real_candidates[c]:
                
                print(checkbox[:,c])
                if c == length-1:
                    continue
                print(len(checkbox[:,c]),checkbox[:,c])
                print(len(checkbox[:,length-1]),checkbox[:,length-1])
                check = checkbox[:,c]&checkbox[:,length-1]
                check_sum = (check).sum()
                print(check, check_sum)
                if check_sum < 1:
                    print("idx: ",candidates[i])
                    
        print("&"*40)
        print("Insgesamt wäre das mapping: ")
        for i in candidates:
            print(i," : ",self.outgoing_connections[i])

        #self.remove_idx(idx)
        self.update_nodes_of_interest()#at the end     
        
        
        
        
        
        
        
        
        
    def pick_last(self):
        last_candidate = len(self.outgoing_connections)-1
        self.check_idx(last_candidate)